# Hosted layer QA
> Runs as an **ArcGIS Online Hosted Notebook** (Standard runtime). `GIS('home')` uses the notebook's own session — no credentials in code. Logic lives in the `agolops` package (tested in CI); cells only orchestrate.

Runs attribute QA against a hosted feature layer using the same rule engine as the ArcGIS Pro toolbox in the companion repo — one set of rules, every runtime.

In [ ]:
# Install the tested logic (a hosted notebook starts from a clean runtime, without git)
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "https://github.com/iDan-Armstrong/arcgis-qa-toolbox/archive/refs/heads/main.zip",
])

In [ ]:
from arcgis.gis import GIS

gis = GIS("home")
LAYER_ITEM_ID = "REPLACE_WITH_ITEM_ID"
RULES = {
    "unique_fields": ["facility_id"],
    "rules": [
        {"kind": "not_null", "fields": ["facility_name", "facility_type"]},
        {"kind": "in_set", "field": "facility_type",
         "values": ["clinic", "hospital", "health_post", "pharmacy"]},
        {"kind": "range", "field": "beds", "min": 0, "max": 500},
    ],
}

In [ ]:
from qacore.rules import Rule, run_rules  # same engine as the Pro toolbox

layer = gis.content.get(LAYER_ITEM_ID).layers[0]
oid_field = layer.properties.objectIdField
fields = sorted({f for r in RULES["rules"] for f in r.get("fields", [r.get("field")])} |
                set(RULES["unique_fields"]))

feature_set = layer.query(out_fields=[oid_field, *fields], return_geometry=False)
records = ((f.attributes[oid_field], {k: f.attributes.get(k) for k in fields})
           for f in feature_set.features)

summary = run_rules(records, [Rule.from_config(r) for r in RULES["rules"]],
                    RULES["unique_fields"])
print("\n".join(summary.to_lines()))

Failing OIDs can be fed straight back into `layer.query` for review, or the summary posted to the team channel. Schedule weekly; pair with the refresh notebook so QA runs the morning after each load.